<img src="./assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Introduction to Regularization

---

## The Bias-Variance Tradeoff
Before we can understand regularization, we must understand the core problem it solves: **overfitting**. Every machine learning model fights a constant battle between two sources of error:

- **Bias:** The error introduced by approximating a complex, real-world problem with a simplified model. In other words, the model is too simple to capture the underlying patterns in the data. A model with high bias fails to find the signal in the data, leading to **underfitting**.

- **Variance:** The error that tells us how much a model's predictions would change if it were trained on a different dataset. A model with high variance pays too much attention to the random noise in the training data, leading to **overfitting**.

<img src="./assets/overfit.png" width=400>

When a model overfits, it often "learns" the training data perfectly by assigning massive _coefficients (weights)_ to features just to capture that noise. **Regularization** is the tool we use to constrain these weights, keeping them small and controlled.

**An Analogy:** Think of an overfitted model like a student who memorizes a sample practice test word-for-word, assuming the actual exam will be identical. When the exam day arrives and the questions are slightly different, the student struggles because they didn't learn the actual concepts—they just memorized the noise.

**The Tradeoff**

Because of how these errors interact, minimizing both simultaneously is nearly impossible. If we try to decrease bias by making our model more flexible, the variance tends to increase. Finding the sweet spot where both bias and variance are minimized is the ultimate goal of machine learning.

**What Causes Overfitting?**

When we lose this balance and lean too far into high variance, overfitting occurs. This typically happens due to a few key culprits:

- **High Model Complexity Relative to Data Size:** If a model has a massive number of features but a relatively small dataset, it has too much freedom. It will easily find accidental patterns in the small sample instead of learning true general rules.

- **Multicollinearity:** When feature columns are highly correlated with each other, they carry redundant information. This confuses the model, making it difficult to determine which feature is truly important, often leading to unstable and inflated weights.

- **Non-Representative or Irrelevant Data:** When a dataset includes "noisy" features that have nothing to do with the main objective, a complex model will still try to find meaning in them, essentially learning relationships that don't exist in the real world.

----

## The Cost Function and The Penalty
To understand how we actually constrain these weights and prevent overfitting, we need to look at how a model learns.

During training, a model's entire goal is to find the coefficients that minimize its total error. Think of a **Cost Function** _(or Loss Function)_ as the model's scorecard. Just like in golf, a lower score is better. The model calculates the difference between its predictions and the actual truth, and then searches for the exact coefficients that bring this scorecard as close to zero as possible.

In standard Linear Regression, the cost function only cares about minimizing that prediction error:
> `Cost = Sum of Squared Errors (Loss)`

**Regularization** changes the rules of the game by adding a **"penalty"** term to this scorecard. Now, the model is forced to balance two competing goals: minimize the prediction error and keep the coefficients small.
> `Regularized Cost = Loss + Penalty based on coefficient size`

By changing the scorecard, the model is penalized if it tries to assign massive weights to features just to chase down random noise.

----

<div class="alert alert-warning">
⚠️ <b>The Golden Rule of Regularization: SCALING</b>
</div>

Before applying any regularization technique, there is one non-negotiable step: **you must scale your data** (for example, using `StandardScaler` within a machine learning `Pipeline`).Because the regularization penalty is calculated directly from the absolute size of the coefficients, features on larger numerical scales will naturally distort the penalty. For instance, consider a model predicting real estate value using two features:
- **House Price/Value:** Scaled in hundreds of thousands (e.g., $\$500,000$)
- **Number of Bedrooms:** Scaled in single digits (e.g., $3$)

_**This is critical because the "penalty rate ($\alpha$)" is applied uniformly to all features, regardless of their actual importance or relevance.**_ If you do not standardize your data first, the model will inherently assign much smaller coefficients to the house price feature just to handle its massive scale. Because **$\alpha$** treats all coefficients the same, the penalty will unfairly wipe out or heavily suppress the bedroom feature while barely affecting the price feature—purely because of the differing units used. Scaling ensures that every feature competes on a level playing field, allowing **$\alpha$** to penalize them fairly based on their true predictive power.

---

## What is Regularization?
At its core, **regularization** is a technique that adds a penalty to the model's cost function to discourage it from learning overly complex patterns. By forcing the model to keep its coefficients small and controlled, we prevent it from memorizing the noise in our training data. This trade-off slightly reduces training accuracy but drastically improves the model's ability to generalize to new, unseen data.

While all regularization techniques share this same goal, they achieve it in slightly different ways.

### The Three Types of Regularization

| Model | Penalty Type | How it Works | When to Use It |
| :--- | :--- | :--- | :--- |
| **Ridge (L2)** | Squares the weights ($\alpha \sum \beta^2$) | Shrinks weights toward zero, but rarely exactly zero. Distributes weights evenly among correlated features. | Your **default choice**. Use when you have many features and want to prevent extreme weights from destabilizing your predictions, especially when your variables are highly correlated. |
| **Lasso (L1)** | Absolute value of weights ($\alpha \sum |\beta|$) | Ruthlessly forces the weights of less important features to **exactly zero**. | When you have hundreds of features and want automated **Feature Selection** (dropping useless columns). |
| **ElasticNet** | Combination of L1 and L2 | Balances both penalties using a ratio. | When you have many correlated features, but still want to drop the completely useless ones. |

*<sub>_**$\alpha$ also referred to as $\lambda$**_</sub>

---


## Keeping Highly Correlated Features in Regression

While multicollinearity can inflate variance, dropping highly correlated features is not always the best choice. Here are some examples where you should keep them in your regression model.

* Here are some real-world scenarios across different fields demonstrating exactly why dropping highly correlated features damages your model.

### 1. Cardiovascular Risk Modeling (Clinical Validity)

*   **The Features:** **Systolic Blood Pressure** ($X_1$) and **Diastolic Blood Pressure** ($X_2$).
*   **The Correlation:** Very high. If a patient’s systolic pressure (the top number) goes up, their diastolic pressure (the bottom number) almost always goes up as well.
*   **Why Keep Both?** 
    *   They measure completely different phases of a heartbeat. 
    *   High systolic blood pressure indicates stiffening arteries.
    *   High diastolic blood pressure indicates high vascular resistance in smaller blood vessels
    *   If you drop Diastolic BP, the model cannot identify patients who have normal Systolic BP but dangerously high Diastolic BP, leading to missed life-threatening cardiovascular risks.

### 2. Real Estate Appraisal (Predictive Accuracy)

*   **The Features:** **Total Square Footage** ($X_1$) and **Number of Bedrooms** ($X_2$).
*   **The Correlation:** Highly correlated because larger houses generally have more bedrooms.
*   **Why Keep Both?** 
    *   If you drop **Number of Bedrooms**, your model cannot differentiate between a $2,000\text{ sq ft}$ open-concept loft and a $2,000\text{ sq ft}$ family home with 4 bedrooms.
    *   Both configurations attract entirely different buyer demographics and command different market prices.

### 3. Epidemiology & Public Health (Causal Integrity)

*   **The Features:** **Average Daily Cigarettes Smoked** ($X_1$) and **Years of Smoking History** ($X_2$).
*   **The Correlation:** Highly correlated because heavy smokers often have long histories of smoking.
*   **Why Keep Both?**
    *   Dropping **Years of Smoking History** creates Omitted Variable Bias, forcing the model to falsely attribute long-term biological duration damage entirely to daily volume.
    *   A person smoking 40 cigarettes a day for 2 years faces vastly different physiological risks than someone smoking 5 cigarettes a day for 30 years.

### 4. Digital Marketing & E-Commerce (Business Logic)

*   **The Features:** **Website Clicks** ($X_1$) and **Impressions/Views** ($X_2$).
*   **The Correlation:** Extremely highly correlated; more ad impressions naturally lead to more clicks.
*   **Why Keep Both?**
    *   Dropping **Impressions** leaves you unable to calculate or control for the Click-Through Rate (CTR).
    *   Both metrics are required by marketing stakeholders to calculate return on ad spend (ROAS) and evaluate ad creative performance.

---


## Let's look at how we can apply regularization in practice.

In [163]:
# --- Setup and Imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Datasets
from sklearn.datasets import load_diabetes

# Preprocessing & Pipelines
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Regression Models and Metrics
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

# Classification Models and Metrics
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

import warnings
warnings.filterwarnings('ignore')

## 1. Regularization in Regression
**Dataset:** The Diabetes Dataset. It has 10 baseline variables, age, sex, BMI, and blood serum measurements. Many of these biological metrics are highly correlated, making it perfect for regularization.

Our Target is a quantitative measure of disease progression one year after the baseline measurements were taken.
Think of it as a numerical **"severity score"** assigned to each patient.
- It is a continuous number that ranges from 25 to 346.
- A higher score means the patient's diabetes progressed significantly over that year, while a lower score means the disease was relatively stable.

In [164]:
# Load Regression Data


# List to store our score metrics


In [ ]:
# Look at a sample of the features


In [ ]:
# Look at the sample of the target


**Note:** The dataset 
Looking at the correlation matrix below, there is significant multicollinearity, specifically among the blood serum measurements (s1 through s6).

Here are the biggest offenders in your table:

- **`s1` and `s2`(0.897):** This is an extremely high positive correlation (almost 0.90). In the real world, these represent different measurements of cholesterol (LDL and HDL), which naturally move together.
- **`s3` and `s4` (-0.738):** This is a highly negative correlation. As one goes up, the other reliably goes down.
- **`s2` and `s4` (0.660):** A moderately high positive correlation.

In [ ]:
plt.subplots(figsize=(10, 6))
sns.heatmap(
    X[['age', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']].corr()
    ,annot=True,fmt=".3f", cmap='RdYlGn', linewidth=.5, vmin=-1, vmax=1)
plt.show()

In [ ]:
# split data to 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Base Model (Linear Regression without Regularization)

#### 1. Using normal way of fitting

In [ ]:
# 1. Initialize and fit the scaler on training features only


# 2. Transform the test features using the already-fitted scaler


# 3. Initialize, fit, and predict with the Linear Regression model



print(f'RMSE: ')  # complete code here
print(f'R2 Score: ') # complete code here

#### 1. Introducing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline

# Base Model (Linear Regression without Regularization)
# create the pipeline


# fit and predict



# append the Model name,RMSE,R2 result to reg_results
reg_results.append({
    'Model': 'Base Linear Regression',
    'RMSE': # complete code here,
    'R2 Score': #complete code here  
})

# show the first appended model


In [ ]:
# check the type of the lr_base_pipe


In [ ]:
# check the lr_base_pipe scaler column 


In [ ]:
# Base Model Coefficient 


### Using `RidgeCV` and `LassoCV`

The difference between the standard models (`Ridge` / `Lasso`) and their **"CV"** counterparts (`RidgeCV` / `LassoCV`) comes down to manual vs. automatic **hyperparameter tuning**.

The **"CV" stands for Cross-Validation**. In Scikit-Learn, these CV classes are specially designed to find the **_best penalty strength (the alpha parameter)_** automatically, saving you from having to build a separate `GridSearchCV`.

**Here is the detailed breakdown of how they differ:**
1. **The Standard Models: `Ridge`, `Lasso` and `ElasticNet`**
When you use the standard `Ridge` or `Lasso` classes, **you must guess the penalty strength manually**. You provide a single value for the **alpha** parameter _(e.g., `alpha=1.0`)_. However, in the case of `ElasticNet` you would also need to guess the `l1_ratio` parameter. The model runs once, applies that exact penalty, and finishes. You will need to reiterate the process to find the best alpha manually, or you would have to wrap the model in a `GridSearchCV` and test them all yourself.
2. **The Auto-Tuning Models: `RidgeCV`, `LassoCV`and `ElasticNetCV`**
When you use `RidgeCV` or `LassoCV`, you are handing the model a list of possible penalty strengths (e.g., `alphas=[0.1, 1.0, 10.0]`). The model will automatically test all of them using cross-validation and pick the best one for you. In the case of `ElasticNetCV` you would also need to provide a list of possible `l1_ratio` values.

**RECAP - How the Cross-Validation Works:**
When you call `.fit()`, the model doesn't just evaluate the **alphas** once. It secretly divides your training data into multiple equal chunks (called "folds"). It trains on a few chunks and tests itself on the remaining chunk, rotating through until every piece of data has been used for validation. By averaging the errors across all these folds, it figures out which alpha produces the most consistently accurate results, and then trains its final self using that winning alpha.

>_**This built-in process is highly optimized, computing the best alpha significantly faster than if you used a standard `GridSearchCV`.**_

**Ridge Regression (L2 Regularization)**

Ridge adds a penalty based on the squared magnitude of the coefficients ($\alpha \sum \beta^2$).
- It shrinks the coefficients of less important features close to zero, but it will almost never force them to exactly zero.
- Datasets where you believe most features contribute something to the prediction, and specifically when you have multicollinearity (highly correlated features). Ridge distributes the weights evenly among correlated variables.
- It keeps all your features but reduces their individual impact to stabilize the model.

In [ ]:
from sklearn.linear_model import RidgeCV

# RidgeCV (L2) specify alpha ranges using np.logspace
alphas = np.logspace(-3, 3, 100) # Generates 100 values evenly spread between 0.001 and 1000 (10⁻³ to 10³)

# create the pipeline


# fit and predict



# append the Model name,RMSE,R2,Best Alpha result to reg_results
reg_results.append({
    'Model': 'Ridge Regression (L2)',
    'RMSE':  # complete code here  ,
    'R2 Score': # complete code here ,
    'Best Alpha': # complete code here
})

# show the second appended model



In [1]:
# RidgeCV Coefficient 



**Lasso Regression (L1 Regularization)**

Lasso adds a penalty based on the absolute value of the coefficients ($\alpha \sum |\beta|$).
- Because of how absolute values calculate the penalty, Lasso is ruthless. It will shrink the coefficients of less important features to exactly zero.
- Datasets with hundreds of features where you suspect many are useless.
- Lasso acts as an automated Feature Selection tool, literally dropping irrelevant columns from your model's decision-making process.

In [ ]:
from sklearn.linear_model import LassoCV

# 3. LassoCV (L1)
# create the pipeline



# fit and predict



reg_results.append({
    'Model': 'Lasso Regression (L1)',
    'RMSE': # complete code here,
    'R2 Score': # complete code here,  # or 'R2 Score': lasso_pipe.score(X_test, y_test)
    'Best Alpha':# complete code here
    
})

# show the third appended model



In [2]:
# LassoCV Coefficient 



**ElasticNet (The Hybrid)**

ElasticNet combines both the **Lasso (L1)** and **Ridge (L2)** penalties into a single equation.
- It uses a hyperparameter (often called the `l1_ratio`__*__) to balance the behavior of Ridge and Lasso.
- Situations where you have a massive dataset with lots of correlated features, and you want the feature selection of Lasso without it randomly dropping correlated features too aggressively.
- It is the safest "best of both worlds" approach, but it requires more computational time to tune both the overall penalty strength and the L1/L2 ratio.

>__*__ The `l1_ratio` is a number that must fall strictly between 0.0 and 1.0. Here is how to think about the values:
>- __`l1_ratio = 0.0` (Pure Ridge):__ The model completely ignores the Lasso penalty. (Note: Scikit-Learn actually recommends just using the Ridge class if you want this, as it calculates faster!)
>- __`l1_ratio = 1.0` (Pure Lasso):__ The model completely ignores the Ridge penalty.
>- __`l1_ratio = 0.5` (50/50 Split):__ The model applies an equal mathematical penalty from both L1 and L2.

>By setting the l1_ratio to something high like 0.95, you are telling the model: __*"Act 95% like Lasso and drop the useless columns, but keep 5% of Ridge's behavior just to act as 'glue' to stabilize the highly correlated features."*__

In [ ]:
from sklearn.linear_model import ElasticNetCV

# 1. Define parameter grids natively for ElasticNetCV
# Note: ElasticNetCV prefers an explicit array format for parameters
alphas_list = [0.01, 0.1, 1.0, 10.0]
l1_ratios_list = [0.2, 0.5, 0.8]

# 2. Build the pipeline with ElasticNetCV inside




# 3. Fit the pipeline (Cross-validation runs internally on the enet_cv step)



# 4. Generate predictions using the best found hyperparameters



# 5. Append results normally
reg_results.append({
    'Model': 'ElasticNetCV',
    'RMSE': # complete code here,
    'R2 Score': # complete code here,
    'Best Alpha': # complete code here
})


# show the fourth appended model


In [ ]:
# Convert the reg_results to pd.DataFrame 
# then Compare Regression Results



## Wrap-Up:

It is completely normal to feel overwhelmed by all these different models and acronyms! The most important thing to remember as a data scientist is this: **You do not need to magically know the perfect model before you start.** Machine learning is about conducting controlled experiments. You start with a simple baseline model. If it overfits, you run a new experiment adding a "penalty" (regularization). You observe the results, tweak the approach, and see what works best for your specific data.

Here is your conceptual guide for setting up those experiments.

### The "Which Approach Do I Choose?" Guide

When deciding which regularization experiment to run, think about what you want to do with your dataset's columns (features):

* **Ridge (L2):** Choose this when you believe all features contribute a small, real effect to the prediction, or when features are highly correlated. Instead of throwing columns away, Ridge evenly shrinks all coefficient weights to stabilize the model and combat multicollinearity. It serves as your safest, most reliable baseline experiment.
* **Lasso (L1):** Choose this when you have a high-dimensional dataset (many columns) and suspect a significant portion of them are irrelevant noise. Lasso acts as a built-in feature selector—its absolute penalty geometry ruthlessly forces the weights of non-predictive variables to exactly zero, generating a clean, highly interpretable sparse model (most of the coefficient weights are set exactly to zero)
* **ElasticNet:** Choose this when you are dealing with a complex, high-dimensional dataset containing many highly correlated features. ElasticNet combines the strengths of both worlds—it leverages Lasso (L1) to ruthlessly eliminate completely useless noise, while utilizing Ridge (L2) to ensure that correlated variables share weight and stabilize the model rather than being arbitrarily deleted.

### Watch Out For:

No matter which experiment you run, if you forget these three rules, your results will be invalid:

1. **You Must Scale Your Data:** Regularization is obsessed with the size of your numbers. If you do not put your data through a `StandardScaler`, a column with large numbers (like Income = $80,000) will be penalized much harder than a column with small numbers (like Age = 25).
2. **Split Before You Scale:** To prevent "Data Leakage" (where your training data accidentally memorizes the test data), you must always split your data into train and test sets *before* you apply your scaler. Using a Pipeline is the easiest way to guarantee you do this correctly.
3. **Opposite Dials (`alpha` vs. `C`):**
    * When running **Regression** (Ridge/Lasso), the penalty dial is called **`alpha`**. Turning it *up* makes the penalty stronger.
    * When running **Classification** (Logistic Regression), the penalty dial is called **`C`**. Turning it *down* makes the penalty stronger.

## References and Further Reading

### Books and Textbooks

1. **Introduction to Statistical Learning (ISLR)** by James, Witten, Hastie, and Tibshirani
   - Chapter 6: Linear Model Selection and Regularization
   - Available free online at: [statlearning.com](https://www.statlearning.com/)
   - Excellent introduction with both theory and practical examples

2. **The Elements of Statistical Learning** by Hastie, Tibshirani, and Friedman
   - Chapter 3: Linear Methods for Regression
   - More mathematically rigorous, great for deeper understanding
   - Available free online at: [web.stanford.edu/~hastie/ElemStatLearn](https://web.stanford.edu/~hastie/ElemStatLearn/)

### Original Research Papers

3. **Ridge Regression**: 
   - Hoerl, A. E., & Kennard, R. W. (1970). "Ridge Regression: Biased Estimation for Nonorthogonal Problems." *Technometrics*, 12(1), 55-67. [doi.org/10.1080/00401706.1970.10488634](https://doi.org/10.1080/00401706.1970.10488634)

4. **Lasso (Least Absolute Shrinkage and Selection Operator)**:
   - Tibshirani, R. (1996). "Regression Shrinkage and Selection via the Lasso." *Journal of the Royal Statistical Society: Series B*, 58(1), 267-288. [doi.org/10.1111/j.2517-6161.1996.tb02080.x](https://doi.org/10.1111/j.2517-6161.1996.tb02080.x)

5. **Elastic Net**:
   - Zou, H., & Hastie, T. (2005). "Regularization and Variable Selection via the Elastic Net." *Journal of the Royal Statistical Society: Series B*, 67(2), 301-320. [doi.org/10.1111/j.1467-9868.2005.00503.x](https://doi.org/10.1111/j.1467-9868.2005.00503.x)
   - This is the foundational paper introducing Elastic Net

### Online Resources

6. **Scikit-learn Documentation**
   - Ridge Regression: [scikit-learn: Ridge Regression](https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression)
   - Lasso: [scikit-learn: Lasso](https://scikit-learn.org/stable/modules/linear_model.html#lasso)
   - Elastic Net: [scikit-learn: Elastic Net](https://scikit-learn.org/stable/modules/linear_model.html#elastic-net)

7. **Stanford CS229 Notes on Regularization**
   - Comprehensive notes on bias-variance tradeoff and regularization
   - Course site: [cs229.stanford.edu](https://cs229.stanford.edu/)

8. **Towards Data Science Articles**
   - Numerous beginner-friendly articles on Medium/Towards Data Science explaining regularization concepts
   - Browse: [towardsdatascience.com](https://towardsdatascience.com/) (search "Ridge vs Lasso", "Elastic Net")

### Interactive Visualizations

9. **Regularization Plot App**
   - Interactive visualization showing how Ridge and Lasso penalties affect coefficients
   - App: [timothykbook.shinyapps.io/RegularizationPlot](https://timothykbook.shinyapps.io/RegularizationPlot/)

### Video Tutorials

10. **StatQuest with Josh Starmer**
    - YouTube channel with excellent visual explanations of Ridge, Lasso, and Elastic Net
    - Channel: [youtube.com/@statquest](https://www.youtube.com/@statquest)


### For Deeper Understanding

13. **Bias-Variance Tradeoff**
    - Understanding this fundamental concept helps explain why regularization works
   - Hastie, Tibshirani, and Friedman cover this extensively in their textbooks

14. **Cross-Validation**
    - Understanding cross-validation is essential for properly tuning regularization parameters
   - Covered in ISLR Chapter 5 and throughout most machine learning resources


